## EDA 및 전처리

## 라이브러리 불러오기

In [1]:
import pandas as pd
import numpy as np
import warnings
import joblib

from sklearn.model_selection import train_test_split

In [2]:
loan_data = pd.read_csv('data/loan_data.csv')
loan_data.head()

,나이,성별,연소득,근속연수,주거형태,신용점수,기존대출건수,연간카드사용액,부채비율,대출신청액,대출목적,상환방식,대출기간,승인여부
0,42,여,4500,16,전세,613,2,3140,28.5,3000,자동차,원금균등,36,1
1,36,남,2400,4,월세,437,3,1260,54.2,2200,주택구입,원리금균등,36,0
2,43,남,4900,15,월세,623,2,3210,28.2,3800,자동차,원리금균등,48,1
3,51,여,2500,22,자가,608,1,1720,35.7,5600,전세자금,원리금균등,36,0
4,35,여,3900,3,월세,549,0,1630,49.5,7300,생활비,원리금균등,24,0


In [3]:
sample_reviews = pd.read_csv('data/sample_reviews.csv')
sample_reviews.head()

,리뷰ID,리뷰텍스트,작성일,상품카테고리,평점
0,R001,주문하고 이틀만에 받았어요. 포장도 꼼꼼하고 상품 상태도 완벽합니다. 재구매 의사 ...,2025-12-01,가전,5
1,R002,배송이 일주일이나 걸렸네요. 거기다 박스가 찌그러져서 왔어요. 상품은 다행히 멀쩡하...,2025-12-02,생활용품,2
2,R003,가격 대비 괜찮은 것 같아요. 엄청 좋진 않지만 이 가격이면 납득할 수 있는 수준이에요.,2025-12-03,의류,3
3,R004,고객센터에 세 번 전화했는데 매번 연결이 안 됩니다. 교환 요청한 지 2주째인데 아...,2025-12-04,가전,1
4,R005,소재가 생각보다 얇아요. 사진이랑 색상도 좀 다르고요. 반품하려다가 귀찮아서 그냥 ...,2025-12-05,의류,2


In [4]:
prediction_logs = pd.read_csv('data/prediction_logs.csv')
prediction_logs.head()

,request_id,timestamp,나이,성별,연소득,근속연수,주거형태,신용점수,기존대출건수,연간카드사용액,부채비율,대출신청액,대출목적,상환방식,대출기간,approved,probability,risk_grade,model_version,latency_ms
0,501d7ce4-560d-41ee-bede-8545040c92bf,2025-04-01 12:29:25,40,남,1900,3,월세,470,1,960,37.8,2800,자동차,원리금균등,24,0,0.1526,D,1.0.0,61.33
1,c65828c7-8c5f-49bc-88ad-dd54945c2ebc,2025-04-01 12:30:05,32,남,5100,4,월세,531,0,1380,20.9,3800,생활비,원금균등,48,1,0.6441,B,1.0.0,16.70
2,81a6fb9c-5fc5-4b53-8c30-aa66f54b50df,2025-04-01 14:58:02,37,여,1600,0,전세,309,1,1000,46.3,2400,자동차,원금균등,48,0,0.0622,D,1.0.0,93.32
3,75c0b780-0aa4-4519-b0e0-9655495278de,2025-04-01 17:13:13,29,남,1600,0,전세,566,4,1040,41.9,1800,생활비,원리금균등,48,0,0.2175,D,1.0.0,47.89
4,197fca37-48e7-4022-8cc8-5838e2b538f9,2025-04-02 09:51:30,23,남,2700,0,자가,401,2,1850,52.9,9400,교육비,원리금균등,48,0,0.0022,D,1.0.0,20.29


In [5]:
loan_data.isnull().sum()

나이         0
성별         0
연소득        0
근속연수       0
주거형태       0
신용점수       0
기존대출건수     0
연간카드사용액    0
부채비율       0
대출신청액      0
대출목적       0
상환방식       0
대출기간       0
승인여부       0
dtype: int64

In [6]:
sample_reviews.isnull().sum()

리뷰ID      0
리뷰텍스트     0
작성일       0
상품카테고리    0
평점        0
dtype: int64

In [7]:
prediction_logs.isnull().sum()

request_id       0
timestamp        0
나이               0
성별               0
연소득              0
근속연수             0
주거형태             0
신용점수             0
기존대출건수           0
연간카드사용액          0
부채비율             0
대출신청액            0
대출목적             0
상환방식             0
대출기간             0
approved         0
probability      0
risk_grade       0
model_version    0
latency_ms       0
dtype: int64

In [8]:
loan_data.describe()

,나이,연소득,근속연수,신용점수,기존대출건수,연간카드사용액,부채비율,대출신청액,대출기간,승인여부
count,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000
mean,38.078667,3884.866667,6.074667,568.626667,1.258000,1757.760000,37.353867,3418.533333,36.840000,0.608667
std,8.582633,1521.267175,5.136927,86.603931,1.136206,995.241538,15.070179,1825.101964,13.990886,0.488211
min,23.000000,1500.000000,0.000000,300.000000,0.000000,280.000000,3.000000,500.000000,12.000000,0.000000
25%,32.000000,2800.000000,2.000000,510.750000,0.000000,1000.000000,25.975000,2200.000000,24.000000,0.000000
50%,38.000000,3650.000000,5.000000,569.000000,1.000000,1590.000000,36.500000,3000.000000,36.000000,1.000000
75%,44.000000,4700.000000,9.000000,626.000000,2.000000,2260.000000,47.825000,4200.000000,48.000000,1.000000
max,65.000000,13900.000000,30.000000,900.000000,5.000000,6620.000000,83.900000,15000.000000,60.000000,1.000000


## 학습/테스트 데이터 분리

### 입력(피쳐) ---> X ---> 타겟을 제외한 나머지 모든 컬럼
### 정답(타겟) ---> y --> 승인여부 (y/n)

In [9]:
df = pd.read_csv('data/loan_data.csv')

In [10]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df['성별'] = label_encoder.fit_transform(df['성별'])
df['성별']

0       1
1       0
2       0
3       1
4       1
       ..
1495    0
1496    0
1497    0
1498    0
1499    1
Name: 성별, Length: 1500, dtype: int64

In [11]:
df['주거형태'] = label_encoder.fit_transform(df['주거형태'])
df['주거형태']

0       2
1       0
2       0
3       1
4       0
       ..
1495    1
1496    1
1497    2
1498    1
1499    0
Name: 주거형태, Length: 1500, dtype: int64

In [12]:
df['대출목적'] = label_encoder.fit_transform(df['대출목적'])
df['대출목적']

0       4
1       6
2       4
3       5
4       2
       ..
1495    2
1496    1
1497    0
1498    6
1499    2
Name: 대출목적, Length: 1500, dtype: int64

In [13]:
df['상환방식'] = label_encoder.fit_transform(df['상환방식'])
df['상환방식']

0       1
1       2
2       2
3       2
4       2
       ..
1495    1
1496    1
1497    2
1498    2
1499    0
Name: 상환방식, Length: 1500, dtype: int64

In [14]:
df.head()

,나이,성별,연소득,근속연수,주거형태,신용점수,기존대출건수,연간카드사용액,부채비율,대출신청액,대출목적,상환방식,대출기간,승인여부
0,42,1,4500,16,2,613,2,3140,28.5,3000,4,1,36,1
1,36,0,2400,4,0,437,3,1260,54.2,2200,6,2,36,0
2,43,0,4900,15,0,623,2,3210,28.2,3800,4,2,48,1
3,51,1,2500,22,1,608,1,1720,35.7,5600,5,2,36,0
4,35,1,3900,3,0,549,0,1630,49.5,7300,2,2,24,0


In [15]:
X = df.drop('승인여부', axis=1)
y = df['승인여부']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [17]:
X_train.shape

(1200, 13)

In [18]:
y_train.shape

(1200,)

### 파이프라인 (스케일링 + 머신러닝 모델링)

In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier

In [20]:
pipeline = Pipeline([
    ('scaler',StandardScaler()),
    ('model', XGBClassifier(
        n_estmators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss'
    ))
])

In [21]:
pipeline

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None


### 학습

In [22]:
pipeline.fit(X_train, y_train)

c:\Users\Administrator\bigdata2026\MLops\.venv\lib\site-packages\xgboost\training.py:200: UserWarning: [15:30:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "n_estmators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None


### 평가

In [23]:
y_pred = pipeline.predict(X_test)

In [24]:
y_pred

array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0,
       1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1,
       0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1,
       0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1,
       0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1])

## 평가 지표 - 정확도

In [25]:
accuracy_score(y_test, y_pred)

0.8366666666666667

### 혼동 행렬

In [26]:
confusion_matrix(y_test, y_pred)

array([[ 94,  29],
       [ 20, 157]])

In [27]:
pipeline.named_steps['model']

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [28]:
xgb_model = pipeline.named_steps['model']
feature_names = X.columns

In [29]:
pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values(ascending=False)

연소득        0.181362
대출신청액      0.144815
근속연수       0.126707
신용점수       0.089497
상환방식       0.082079
주거형태       0.069323
나이         0.068888
기존대출건수     0.059058
부채비율       0.044618
연간카드사용액    0.043996
대출목적       0.032890
대출기간       0.032221
성별         0.024547
dtype: float32